In [2]:
import numpy as np
import pandas as pd
import gymnasium as gym
from typing import Optional
import math
import copy
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [3]:
# products with most different price points over history
PRODUCT_LIST = ["FOODS_1_096","FOODS_1_129"]
CHOSEN_STATE="TX"

Load data and transform

In [4]:
unit_sales = pd.read_csv("../data/sales_train_evaluation.csv")
prices = pd.read_csv("../data/sell_prices.csv")
calendar = pd.read_csv("../data/calendar.csv")

In [5]:
# get product-level demand for FOODS per day
# from EDA, the prices were founds to be state-dependent
foods_demand = unit_sales[unit_sales.state_id == CHOSEN_STATE]
foods_demand = foods_demand.loc[foods_demand.cat_id == 'FOODS',["item_id"] + [col for col in foods_demand.columns if col.startswith("d_")]]
q = foods_demand.groupby(by=["item_id"]).mean().reset_index()
# filter for products
q = q[q.item_id.isin(PRODUCT_LIST)]

merged_df = calendar.merge(prices, how="left", on="wm_yr_wk")
merged_df = merged_df[merged_df.item_id.isin(PRODUCT_LIST)]

# obtain weekly prices
p = (
    merged_df
    .groupby(by=["item_id", "d"])["sell_price"]
    .mean()
    .reset_index()
    .assign(d_num=lambda df: df["d"].str.extract(r'(\d+)').astype(int))
    .sort_values(["item_id", "d_num"])
    .drop(columns="d_num")
)

q_long = q.reset_index().melt(id_vars="item_id", var_name="d", value_name="quantity")

pq = p.merge(q_long,how="left", on=["d","item_id"])
pq = pq.dropna()

pq["log_sell_price"] = np.log(pq["sell_price"])

# add SNAP day flag and create separate data frames
pq = pq.merge(calendar[["d",f"snap_{CHOSEN_STATE}"]],how="left",on="d")
pq_snap = pq.loc[pq[f"snap_{CHOSEN_STATE}"]==1].copy()
pq_nonsnap = pq.loc[pq[f"snap_{CHOSEN_STATE}"]==0].copy()

In [6]:
pq["revenue"] = pq["quantity"] * pq["sell_price"]
max_daily_revenue = np.max(pq.groupby(by="d")["revenue"].mean())

In [7]:
max_daily_revenue[0]

IndexError: invalid index to scalar variable.

In [6]:
np.max(pq["quantity"])

np.float64(26.0)

In [5]:
pq.head()

,item_id,d,sell_price,quantity,log_sell_price,snap_TX
0,FOODS_1_096,d_1,7.12,2.333333,1.962908,0
1,FOODS_1_096,d_2,7.12,2.666667,1.962908,0
2,FOODS_1_096,d_3,7.12,3.666667,1.962908,0
3,FOODS_1_096,d_4,7.12,2.333333,1.962908,1
4,FOODS_1_096,d_5,7.12,3.666667,1.962908,0


## Multiple Linear Regression (MLR) using squared terms and gap from reference price

In [ ]:
def _to_day_index(frame):
    """d_1, d_2, ... -> integer index, numerically sorted. No reindex."""
    day = frame.index.str.extract(r"(\d+)", expand=False).astype(int)
    return frame.set_axis(pd.Index(day, name="day")).sort_index()

def estimate_demand_model(df, product_list, window=7, center=True):
    """Quantity on log prices (second-order) plus 7-day reference-price gaps."""

    log_p = _to_day_index(
        df.pivot_table(index="d", columns="item_id",
                       values="log_sell_price", aggfunc="mean")
    )
    q = _to_day_index(
        df.pivot_table(index="d", columns="item_id",
                       values="quantity", aggfunc="mean")
    )

    # lag-1 demand as regressor
    lag_cols = [f"lag1_{col}" for col in q.columns]
    for col in q.columns:
        q[f"lag1_{col}"] = q[col].shift(1)

    X = log_p.add_prefix("log_p_")
    if center:
        X = X - X.mean()

    price_cols = list(X.columns)
    
    mains = " + ".join(f"Q('{c}')" for c in price_cols)
    squares = " + ".join(f"I(Q('{c}')**2)" for c in price_cols)
    lags = " + ".join(f"Q('{c}')" for c in lag_cols)
    formula_rhs = f"({mains})**2 + {squares} + {lags}"

    X = X.join(q[lag_cols])
    params, results_dict = {}, {}

    for product in product_list:
        print(f"--- Fitting AIDS model for {product} ---")
        data = X.copy()
        data["y"] = q[product]
        data = data.dropna()

        res = smf.ols(f"y ~ {formula_rhs}", data=data).fit()
        params[product] = res.params
        results_dict[product] = res
        print(res.summary())

    return pd.DataFrame(params), results_dict

In [8]:
nonsnap_aids_results, results_dict = estimate_demand_model(pq_nonsnap, PRODUCT_LIST, window=7, center=False)

--- Fitting AIDS model for FOODS_1_096 ---
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.638
Model:                            OLS   Adj. R-squared:                  0.636
Method:                 Least Squares   F-statistic:                     325.1
Date:                Thu, 23 Jul 2026   Prob (F-statistic):          1.13e-279
Time:                        18:44:55   Log-Likelihood:                -3366.3
No. Observations:                1300   AIC:                             6749.
Df Residuals:                    1292   BIC:                             6790.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------

In [ ]:
snap_aids_results = estimate_demand_model(pq_snap, PRODUCT_LIST)

--- Fitting AIDS model for FOODS_1_096 ---
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.619
Model:                            OLS   Adj. R-squared:                  0.614
Method:                 Least Squares   F-statistic:                     146.2
Date:                Wed, 22 Jul 2026   Prob (F-statistic):          1.33e-127
Time:                        20:53:30   Log-Likelihood:                -1677.7
No. Observations:                 639   AIC:                             3371.
Df Residuals:                     631   BIC:                             3407.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------